In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,classification_report,ConfusionMatrixDisplay)

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("data/campuspulse_akgec_cleaned.csv")

print(df.shape)

(200000, 37)


In [3]:
df["complaint_date"] = pd.to_datetime(df["complaint_date"], errors="coerce")

df["complaint_month"] = df["complaint_date"].dt.month
df["complaint_dayofweek"] = df["complaint_date"].dt.dayofweek
df["complaint_year"] = df["complaint_date"].dt.year

df["complaint_time"] = pd.to_datetime(df["complaint_time"].astype(str),errors="coerce")

df["complaint_hour"] = df["complaint_time"].dt.hour

print(df.shape)

C:\Users\acer\AppData\Local\Temp\ipykernel_25692\2025595703.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["complaint_time"] = pd.to_datetime(df["complaint_time"].astype(str),errors="coerce")


(200000, 41)


In [4]:
drop_cols = ["complaint_id","complaint_date","complaint_time","complaint_status","response_time_hours","resolution_time_hours",
             "maintenance_team","resolution_type","actual_expenditure"]

df = df.drop(columns=drop_cols)

In [5]:
features = [
    "complainant_type",
    "department",
    "year_of_study",
    "campus_zone",
    "building",
    "floor",
    "room_lab_no",
    "facility_type",
    "class_section",
    "issue_category",
    "issue_subcategory",
    "severity",
    "affected_users",
    "issue_duration_hours",
    "previous_similar_complaints",
    "complaints_last_7_days",
    "complaints_last_30_days",
    "recurrence_count",
    "equipment_age_years",
    "infrastructure_age_years",
    "connectivity_impact",
    "safety_risk",
    "academic_impact",
    "operational_impact",
    "estimated_repair_cost",
    "weather_condition",
    "submission_channel",
    "complaint_month",
    "complaint_dayofweek",
    "complaint_year",
    "complaint_hour"
]

In [6]:
X = df[features]
y = df["priority"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

In [8]:
print(X_train.shape)
print(X_test.shape)

(160000, 31)
(40000, 31)


In [9]:
num_cols = X_train.select_dtypes(include=np.number).columns

cat_cols = X_train.select_dtypes(include="object").columns

C:\Users\acer\AppData\Local\Temp\ipykernel_25692\2394940874.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns


In [10]:
print("Numeric:", len(num_cols))
print("Categorical:", len(cat_cols))

Numeric: 14
Categorical: 17


In [11]:
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])

In [12]:
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(handle_unknown="ignore"))])

In [13]:
preprocessor = ColumnTransformer([("num", num_pipe, num_cols),("cat", cat_pipe, cat_cols)])

## Logistic Regression

In [14]:
lr_model = Pipeline([("preprocessor", preprocessor),("model", LogisticRegression(max_iter=1000,class_weight="balanced"))])

In [15]:
lr_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](4,)","['Critical','High','Low','Medium']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](31,)","['complainant_type','department','year_of_study',...,'complaint_dayofweek', 'complaint_year','complaint_hour']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,31
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (d

In [16]:
y_pred_lr = lr_model.predict(X_test)

## Decision Tree

In [17]:
dt_model = Pipeline([("preprocessor", preprocessor),("model", DecisionTreeClassifier(random_state=42,class_weight="balanced"))])

In [18]:
dt_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](4,)","['Critical','High','Low','Medium']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](31,)","['complainant_type','department','year_of_study',...,'complaint_dayofweek', 'complaint_year','complaint_hour']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,31
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (d

In [19]:
y_pred_dt = dt_model.predict(X_test)

## Random Forest

In [20]:
rf_model = Pipeline([("preprocessor", preprocessor),("model", RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1,class_weight="balanced"))
])

In [21]:
rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](4,)","['Critical','High','Low','Medium']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](31,)","['complainant_type','department','year_of_study',...,'complaint_dayofweek', 'complaint_year','complaint_hour']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,31
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (d

In [22]:
y_pred_rf = rf_model.predict(X_test)

## XGBoost

In [23]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_train_xgb = encoder.fit_transform(y_train)
y_test_xgb = encoder.transform(y_test)

In [24]:
print(encoder.classes_)

['Critical' 'High' 'Low' 'Medium']


In [25]:
xgb_model = Pipeline([("preprocessor", preprocessor),("model", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="mlogloss"
    ))
])

In [26]:
xgb_model.fit(X_train,y_train_xgb)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](4,)","[0,1,2,3]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](31,)","['complainant_type','department','year_of_study',...,'complaint_dayofweek', 'complaint_year','complaint_hour']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,31
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By s

In [27]:
y_pred_xgb_num = xgb_model.predict(X_test)

y_pred_xgb = encoder.inverse_transform(y_pred_xgb_num.astype(int))

In [28]:
def evaluate_model(name, y_true, y_pred):
    return {"Model": name,"Accuracy": accuracy_score(y_true, y_pred),"Precision": precision_score(y_true, y_pred, average="weighted"),
            "Recall": recall_score(y_true, y_pred, average="weighted"),
             "F1": f1_score(
            y_true, y_pred, average="weighted")
    }

In [29]:
results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, y_pred_lr),
    evaluate_model("Decision Tree", y_test, y_pred_dt),
    evaluate_model("Random Forest", y_test, y_pred_rf),
    evaluate_model("XGBoost", y_test, y_pred_xgb)
])

results = results.sort_values("F1", ascending=False).reset_index(drop=True)
results

,Model,Accuracy,Precision,Recall,F1
0,XGBoost,0.650275,0.670509,0.650275,0.650411
1,Random Forest,0.645650,0.659612,0.645650,0.645782
2,Logistic Regression,0.629400,0.642151,0.629400,0.625209
3,Decision Tree,0.511900,0.514098,0.511900,0.512915


In [30]:
lr_params = {"model__C": [0.1, 1, 10],"model__solver": ["lbfgs", "liblinear"]}

lr_grid = GridSearchCV(lr_model,lr_params,cv=3,scoring="f1_weighted",n_jobs=-1)

lr_grid.fit(X_train, y_train)

print("Best LR:", lr_grid.best_params_)
print("Best F1:", lr_grid.best_score_)

c:\Users\acer\Documents\FinalTask\CampusPulse\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:493: FitFailedWarning: 
9 fits failed out of a total of 18.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\acer\Documents\FinalTask\CampusPulse\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\acer\Documents\FinalTask\CampusPulse\.venv\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\acer\Documents\FinalTas

Best LR: {'model__C': 0.1, 'model__solver': 'lbfgs'}
Best F1: 0.6223412817817924


In [31]:
lr_best = lr_grid.best_estimator_

In [32]:
dt_params = {"model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 10, 20],
    "model__min_samples_leaf": [1, 5, 10]
}

dt_grid = GridSearchCV(dt_model,dt_params,cv=3,scoring="f1_weighted",n_jobs=-1)

dt_grid.fit(X_train, y_train)

print("Best DT:", dt_grid.best_params_)
print("Best F1:", dt_grid.best_score_)

Best DT: {'model__max_depth': 5, 'model__min_samples_leaf': 10, 'model__min_samples_split': 2}
Best F1: 0.6329380264003556


In [33]:
dt_best = dt_grid.best_estimator_

In [ ]:
rf_params = {"model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 5]
}

rf_grid = GridSearchCV(rf_model,rf_params,cv=3,scoring="f1_weighted",n_jobs=-1)

rf_grid.fit(X_train, y_train)

print("Best RF:", rf_grid.best_params_)
print("Best F1:", rf_grid.best_score_)

In [ ]:
rf_best = rf_grid.best_estimator_

In [ ]:
xgb_params = {"model__n_estimators": [200, 300],
    "model__max_depth": [3, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0]
}

xgb_grid = GridSearchCV(xgb_model,xgb_params,cv=3,scoring="f1_weighted",n_jobs=-1)

xgb_grid.fit(X_train, y_train_xgb)

print("Best XGB:", xgb_grid.best_params_)
print("Best F1:", xgb_grid.best_score_)

In [ ]:
xgb_best = xgb_grid.best_estimator_

In [ ]:
predictions_tuned = {
    "Logistic Regression": lr_best.predict(X_test),
    "Decision Tree": dt_best.predict(X_test),
    "Random Forest": rf_best.predict(X_test),
    "XGBoost": encoder.inverse_transform(
        xgb_best.predict(X_test).astype(int)
    )
}

In [ ]:
tuned_results = pd.DataFrame([evaluate_model(name, y_test, pred) for name, pred in predictions_tuned.items()])

tuned_results = tuned_results.sort_values("F1", ascending=False).reset_index(drop=True)

tuned_results

In [ ]:
print("Baseline Results")
display(results.sort_values("F1", ascending=False))

print("Tuned Results")
display(tuned_results.sort_values("F1", ascending=False))

In [ ]:
for name, pred in predictions_tuned.items():
    print("\n" + "="*50)
    print(name)
    print("="*50)
    print(classification_report(y_test, pred))

In [ ]:
for name, pred in predictions_tuned.items():
    report = classification_report(y_test, pred, output_dict=True)
    
    print(name,"Critical Recall:", round(report["Critical"]["recall"], 3), "Critical F1:",round(report["Critical"]["f1-score"], 3))

In [ ]:
best_model_name = tuned_results.loc[tuned_results["F1"].idxmax(), "Model"]

print("Best Model:", best_model_name)

In [ ]:
best_pred = predictions_tuned[best_model_name]

ConfusionMatrixDisplay.from_predictions(y_test,best_pred,xticks_rotation=45)

plt.title(f"{best_model_name} - Tuned Confusion Matrix")
plt.show()

In [ ]:
import joblib

model_map = {"Logistic Regression": lr_best, "Decision Tree": dt_best,"Random Forest": rf_best,"XGBoost": xgb_best}

final_model = model_map[best_model_name]

joblib.dump(final_model, "../models/final_priority_model.pkl")

joblib.dump(encoder,"../models/priority_encoder.pkl")

print("Saved:", best_model_name)

In [ ]:
def predict_priority(complaint):
    data = pd.DataFrame([complaint])

    prediction = final_model.predict(data)

    if best_model_name == "XGBoost":
        prediction = encoder.inverse_transform(prediction.astype(int))

    return prediction[0]

In [ ]:
sample = X_test.iloc[0].to_dict()

prediction = predict_priority(sample)

print("Predicted Priority:", prediction)

In [ ]:
probability = final_model.predict_proba(pd.DataFrame([sample]))[0]

if best_model_name == "XGBoost":
    classes = encoder.classes_
else:
    classes = final_model.classes_

print(dict(zip(classes, probability.round(3))))